In [ ]:
import time
import re
from typing import Optional

import pandas as pd
from urllib.parse import quote_plus

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# ⚙ 설정값
# =========================

INPUT_EXCEL = "category.xlsx"                     # 소분류 목록 엑셀
OUTPUT_EXCEL = "gangnamunni_event_urls.xlsx"      # 최종 결과 저장 파일

COL_BIG = "대분류"
COL_MID = "중분류"
COL_SMALL = "소분류"                              # 검색에 사용할 컬럼

# 강남언니 이벤트 검색 기본 URL
GANGNAMUNNI_EVENTS_BASE = "https://www.gangnamunni.com/events"

# 지역 코드
REGIONS = {
    "서울": "N11",
    "경기": "N12",
}

PAGE_LOAD_TIMEOUT = 30
WAIT_AFTER_LOAD = 1.0      # 페이지 로드 후 대기 (초)
HEADLESS = False

SCROLL_MAX_STEP = 120          # 최대 스크롤 횟수
SCROLL_SLEEP = 0.3             # 스크롤 간 대기 시간(초)
NO_GAIN_LIMIT = 4              # 카드 증가가 없는 반복 횟수 허용
EMPTY_ROUND_LIMIT = 2          # 카드가 0개인 상태 허용 횟수

MAX_CARDS_PER_KEYWORD_REGION = 1000   # 한 (키워드, 지역) 당 최대 수집 개수


# =========================
# 공통 유틸
# =========================

def setup_driver(headless: bool = False) -> webdriver.Chrome:
    options = Options()
    options.add_argument("--window-size=1400,900")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-blink-features=AutomationControlled")
    # DOM 준비되면 바로 리턴 → 조금 더 빠름
    options.page_load_strategy = "eager"
    if headless:
        options.add_argument("--headless=new")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.set_script_timeout(PAGE_LOAD_TIMEOUT)
    return driver


def build_gangnamunni_search_url(keyword: str, region_code: str) -> str:
    q = quote_plus(keyword)
    return f"{GANGNAMUNNI_EVENTS_BASE}?q={q}&district={region_code}&s=HIGHEST_RATING_COUNT"


def wait_for_any_card(driver: webdriver.Chrome, timeout: int = 8) -> bool:
    """검색 결과에서 카드가 하나라도 뜰 때까지 대기"""
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "a[id^='event-card-component-ui-']")
            )
        )
        return True
    except TimeoutException:
        return False


def click_load_more_once(driver: webdriver.Chrome, timeout: int = 5) -> None:
    """첫 화면에서 '더보기' 버튼 1회만 확실하게 클릭"""
    btn = None
    try:
        btn = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "button[aria-label='더보기']")
            )
        )
    except TimeoutException:
        try:
            btn = WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located(
                    (By.XPATH, "//button[contains(., '더보기')]")
                )
            )
        except TimeoutException:
            print("   (i) '더보기' 버튼을 찾지 못함 → 클릭 생략")
            return

    if btn is None:
        print("   (i) '더보기' 버튼 객체 없음 → 클릭 생략")
        return

    try:
        if btn.is_displayed() and btn.is_enabled():
            try:
                btn.click()
            except Exception:
                driver.execute_script("arguments[0].click();", btn)
            print("   → '더보기' 버튼 클릭 완료")
            time.sleep(1.0)
    except Exception as e:
        print(f"   (i) '더보기' 클릭 중 예외: {e}")


def click_load_more_if_exists(driver: webdriver.Chrome) -> bool:
    """스크롤 중간에 '더보기' 버튼이 또 나타나면 클릭"""
    clicked = False
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, "button[aria-label='더보기']")
        if not btns:
            btns = driver.find_elements(By.XPATH, "//button[contains(., '더보기')]")

        for btn in btns:
            try:
                if btn.is_displayed() and btn.is_enabled():
                    try:
                        btn.click()
                    except Exception:
                        driver.execute_script("arguments[0].click();", btn)
                    print("   → 스크롤 중 '더보기' 버튼 클릭")
                    time.sleep(0.7)
                    clicked = True
                    break
            except Exception:
                continue
    except Exception:
        pass
    return clicked


def get_event_count(driver: webdriver.Chrome) -> Optional[int]:
    """
    검색 결과 상단의 '284' 숫자를 가져온다.

    1차: h1 안의 span 중 숫자만 있는 것 찾기
    2차: 페이지 전체에서 '숫자+건' 패턴 찾기
    """
    # 1차: h1 span
    try:
        spans = driver.find_elements(By.CSS_SELECTOR, "h1 span")
        for sp in spans:
            text = sp.text.strip()
            if re.match(r"^\d[\d,]*$", text):
                return int(text.replace(",", ""))
    except Exception:
        pass

    # 2차: "숫자 건" 패턴
    try:
        elems = driver.find_elements(By.XPATH, "//*[contains(., '건')]")
        for el in elems:
            text = el.text.strip()
            m = re.search(r"([0-9,]+)\s*건", text)
            if m:
                return int(m.group(1).replace(",", ""))
    except Exception:
        pass

    return None


def collect_gangnamunni_event_cards(
    driver: webdriver.Chrome,
    region_name: str,
    target_total: Optional[int],
):
    """
    - main#frameMain(있으면)을 스크롤하면서
    - a[id^='event-card-component-ui-'] 의 href / text를 전부 수집.
    - target_total(예: 284)이 주어지면 그 개수 이상 모이면 중단.
    """
    collected = {}   # href -> text
    last_count = 0
    no_gain = 0
    empty_rounds = 0

    # 실제 필요한 최대 개수
    if target_total is not None:
        max_needed = min(target_total, MAX_CARDS_PER_KEYWORD_REGION)
    else:
        max_needed = MAX_CARDS_PER_KEYWORD_REGION

    for step in range(1, SCROLL_MAX_STEP + 1):
        anchors = driver.find_elements(By.CSS_SELECTOR, "a[id^='event-card-component-ui-']")
        for a in anchors:
            href = a.get_attribute("href")
            if not href:
                continue
            text = a.text or ""
            collected[href] = text

        cur = len(collected)
        gain = cur - last_count
        print(f"   ▸ Scroll {step:03d} / cards: {cur} (+{gain}) [{region_name}]")

        # 목표 개수 이상이면 즉시 종료
        if cur >= max_needed:
            print(f"   🎯 필요 개수({max_needed}) 이상 수집 완료 → 스크롤 중단")
            break

        # 증가 여부 체크
        if gain == 0:
            no_gain += 1
        else:
            no_gain = 0

        if cur == 0:
            empty_rounds += 1
        else:
            empty_rounds = 0

        last_count = cur

        # 증가 없음 + 이미 충분히 모았으면 종료
        if no_gain >= NO_GAIN_LIMIT:
            if target_total is None or cur >= max_needed:
                print("   ✅ 더 이상 새 카드 없음 → 스크롤 중단")
                break

        # 카드가 계속 0개이면 포기
        if empty_rounds >= EMPTY_ROUND_LIMIT:
            print("   ⚪ 카드가 전혀 안 보임 → 스킵")
            break

        # 🔽 실제 스크롤: main#frameMain 우선, 없으면 window
        driver.execute_script("""
            const main = document.querySelector('main#frameMain');
            const scroller = main || window;
            scroller.scrollBy(0, scroller.clientHeight * 0.9);
        """)

        click_load_more_if_exists(driver)
        time.sleep(SCROLL_SLEEP)

    filtered = []
    for href, text in collected.items():
        filtered.append({"href": href, "card_text": text})

    return filtered


# =========================
# 메인 로직
# =========================

def crawl_gangnamunni_event_urls():
    df_cat = pd.read_excel(INPUT_EXCEL)
    print(f"📂 카테고리 로드: {df_cat.shape[0]} rows\n")

    has_big = COL_BIG in df_cat.columns
    has_mid = COL_MID in df_cat.columns

    if COL_SMALL not in df_cat.columns:
        raise ValueError(f"엑셀에 '{COL_SMALL}' 컬럼이 없습니다. 컬럼명을 확인해 주세요.")

    driver = setup_driver(headless=HEADLESS)
    all_rows = []

    try:
        for idx, row in df_cat.iterrows():
            small = str(row[COL_SMALL]).strip()
            if not small:
                continue

            big = str(row[COL_BIG]).strip() if has_big else None
            mid = str(row[COL_MID]).strip() if has_mid else None

            print(f"\n=== [{idx+1}/{len(df_cat)}] {big} / {mid} / {small} ===")

            for region_name, region_code in REGIONS.items():
                search_url = build_gangnamunni_search_url(small, region_code)
                print(f"🔍 [{region_name}] 검색 URL: {search_url}")

                # 1) 페이지 로드
                loaded = False
                for attempt in range(3):
                    try:
                        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
                        driver.get(search_url)
                        time.sleep(WAIT_AFTER_LOAD)
                        loaded = True
                        break
                    except TimeoutException:
                        print(f"⏰ 검색 페이지 로드 지연 — 재시도 {attempt+1}/3")
                        try:
                            driver.execute_script("window.stop();")
                        except Exception:
                            pass
                        time.sleep(2)

                if not loaded:
                    print(f"🚫 [{region_name}] 검색 페이지 로드 실패 → 스킵")
                    continue

                # 2) 카드 뜰 때까지 대기
                if not wait_for_any_card(driver, timeout=8):
                    print(f"   ❌ [{region_name}] 카드가 하나도 안 뜸 → 스킵")
                    continue

                # 3) 이벤트 총 건수 파싱
                total_events = get_event_count(driver)
                print(f"   ℹ [{region_name}] 검색 이벤트 수: {total_events}")

                # 4) '더보기' 버튼 1회 클릭
                click_load_more_once(driver)

                # 5) 카드 수집
                cards = collect_gangnamunni_event_cards(
                    driver, region_name, target_total=total_events
                )
                print(f"   🎯 '{small}' [{region_name}] 수집된 이벤트: {len(cards)}개")

                if not cards:
                    continue

                for c in cards:
                    href = c["href"]
                    if href.startswith("/"):
                        event_url = "https://www.gangnamunni.com" + href
                    else:
                        event_url = href

                    all_rows.append({
                        "대분류": big,
                        "중분류": mid,
                        "소분류": small,
                        "region": region_name,
                        "gangnamunni_search_url": search_url,
                        "search_event_count": total_events,
                        "event_url": event_url,
                        "card_text": c["card_text"],
                    })

            # 소분류 하나 끝날 때마다 중간 저장
            tmp = pd.DataFrame(all_rows)
            try:
                tmp.to_excel("gangnamunni_event_urls_tmp.xlsx",
                             index=False, engine="openpyxl")
                print(f"💾 중간 저장: gangnamunni_event_urls_tmp.xlsx "
                      f"(현재 {tmp.shape[0]}행)")
            except PermissionError:
                print("⚠ 중간 저장 실패: gangnamunni_event_urls_tmp.xlsx 가 "
                      "다른 프로그램에서 열려 있습니다. 엑셀을 닫으면 다음부터 저장됩니다.")

    finally:
        driver.quit()

    df_out = pd.DataFrame(all_rows)
    df_out.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\n✅ 완료 → {OUTPUT_EXCEL} (총 {df_out.shape[0]}행)")

    return df_out


if __name__ == "__main__":
    crawl_gangnamunni_event_urls()


In [19]:
import time
import re
import pandas as pd
from urllib.parse import quote_plus

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# ⚙ 설정값
# =========================

INPUT_EXCEL = "category_529~622.xlsx"                     # 소분류 목록 엑셀
OUTPUT_EXCEL = "gangnamunni_event_urls.xlsx"      # 최종 결과 저장 파일

COL_BIG = "대분류"
COL_MID = "중분류"
COL_SMALL = "소분류"                              # 검색에 사용할 컬럼

GANGNAMUNNI_EVENTS_BASE = "https://www.gangnamunni.com/events"

# 지역코드
REGIONS = {
    "서울": "N11",
    "경기": "N12",
}

PAGE_LOAD_TIMEOUT = 30
WAIT_AFTER_LOAD = 1.0      # 페이지 로드 후 대기
HEADLESS = False

SCROLL_MAX_STEP = 120       # 최대 스크롤 횟수
SCROLL_SLEEP = 0.3          # 스크롤 간 대기 시간
NO_GAIN_LIMIT = 4           # 카드 증가가 없는 반복 허용 횟수
EMPTY_ROUND_LIMIT = 2       # 카드가 0개인 상태 허용 횟수

MAX_CARDS_PER_KEYWORD_REGION = 1000   # 한 (키워드, 지역)당 최대 수집 개수


# =========================
# 공통 유틸
# =========================

def setup_driver(headless: bool = False):
    options = Options()
    options.add_argument("--window-size=1400,900")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-blink-features=AutomationControlled")
    # DOM만 준비되면 바로 넘기도록 → 약간 더 빠름
    options.page_load_strategy = "eager"
    if headless:
        options.add_argument("--headless=new")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    driver.set_script_timeout(PAGE_LOAD_TIMEOUT)
    return driver


def build_gangnamunni_search_url(keyword: str, region_code: str) -> str:
    q = quote_plus(keyword)
    return f"{GANGNAMUNNI_EVENTS_BASE}?q={q}&district={region_code}&s=HIGHEST_RATING_COUNT"


def wait_for_any_card(driver, timeout: int = 8) -> bool:
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "a[id^='event-card-component-ui-']")
            )
        )
        return True
    except TimeoutException:
        return False


def click_load_more_once(driver, timeout: int = 5):
    """첫 화면에서 '더보기' 버튼 1회만 확실하게 클릭"""
    btn = None
    try:
        btn = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "button[aria-label='더보기']")
            )
        )
    except TimeoutException:
        try:
            btn = WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located(
                    (By.XPATH, "//button[contains(., '더보기')]")
                )
            )
        except TimeoutException:
            print("   (i) '더보기' 버튼을 찾지 못함 → 클릭 생략")
            return

    if btn is None:
        print("   (i) '더보기' 버튼 객체 없음 → 클릭 생략")
        return

    try:
        if btn.is_displayed() and btn.is_enabled():
            try:
                btn.click()
            except Exception:
                driver.execute_script("arguments[0].click();", btn)
            print("   → '더보기' 버튼 클릭 완료")
            time.sleep(1.0)
    except Exception as e:
        print(f"   (i) '더보기' 클릭 중 예외: {e}")


def click_load_more_if_exists(driver) -> bool:
    """스크롤 중 '더보기'가 다시 나타나면 클릭"""
    clicked = False
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, "button[aria-label='더보기']")
        if not btns:
            btns = driver.find_elements(By.XPATH, "//button[contains(., '더보기')]")

        for btn in btns:
            try:
                if btn.is_displayed() and btn.is_enabled():
                    try:
                        btn.click()
                    except Exception:
                        driver.execute_script("arguments[0].click();", btn)
                    print("   → 스크롤 중 '더보기' 버튼 클릭")
                    time.sleep(0.7)
                    clicked = True
                    break
            except Exception:
                continue
    except Exception:
        pass
    return clicked


def get_event_count(driver):
    """
    검색 결과 상단의 '284' 숫자를 가져온다.

    1차: h1 안에 있는 span 중 '숫자만 들어 있는 span' 찾기
    2차: 페이지 전체에서 '숫자+건' 패턴 텍스트 찾기
    """
    # 1차: h1 span에서 숫자만 있는 것 찾기
    try:
        spans = driver.find_elements(By.CSS_SELECTOR, "h1 span")
        for sp in spans:
            text = sp.text.strip()
            if re.match(r"^\d[\d,]*$", text):
                return int(text.replace(",", ""))
    except Exception:
        pass

    # 2차: '숫자+건' 패턴
    try:
        elems = driver.find_elements(By.XPATH, "//*[contains(., '건')]")
        for el in elems:
            text = el.text.strip()
            m = re.search(r"([0-9,]+)\s*건", text)
            if m:
                return int(m.group(1).replace(",", ""))
    except Exception:
        pass

    return None


def collect_gangnamunni_event_cards(driver, region_name: str, target_total: int | None):
    """
    - window를 계속 스크롤
    - a[id^='event-card-component-ui-'] 의 href / text 전부 수집
    - target_total(예: 284)이 주어지면 그 개수 이상 모이면 바로 종료
    """
    collected = {}   # href -> text
    last_count = 0
    no_gain = 0
    empty_rounds = 0

    # 실제로 필요한 최대 개수 (이벤트 수와 상한값 중 작은 값)
    if target_total is not None:
        max_needed = min(target_total, MAX_CARDS_PER_KEYWORD_REGION)
    else:
        max_needed = MAX_CARDS_PER_KEYWORD_REGION

    for step in range(1, SCROLL_MAX_STEP + 1):
        anchors = driver.find_elements(By.CSS_SELECTOR, "a[id^='event-card-component-ui-']")
        for a in anchors:
            href = a.get_attribute("href")
            if not href:
                continue
            text = a.text or ""
            collected[href] = text

        cur = len(collected)
        gain = cur - last_count
        print(f"   ▸ Scroll {step:03d} / cards: {cur} (+{gain}) [{region_name}]")

        # 목표 개수 이상이면 바로 중단
        if cur >= max_needed:
            print(f"   🎯 필요 개수({max_needed}) 이상 수집 완료 → 스크롤 중단")
            break

        if gain == 0:
            no_gain += 1
        else:
            no_gain = 0

        if cur == 0:
            empty_rounds += 1
        else:
            empty_rounds = 0

        last_count = cur

        # 여기서 수정: 목표 개수에 한참 못 미치면 no_gain 이라도 계속 스크롤
        if no_gain >= NO_GAIN_LIMIT:
            if target_total is None or cur >= max_needed:
                print("   ✅ 더 이상 새 카드 없음 → 스크롤 중단")
                break

        if empty_rounds >= EMPTY_ROUND_LIMIT:
            print("   ⚪ 카드가 전혀 안 보임 → 스킵")
            break

        # window 기준으로 스크롤
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.9);")

        click_load_more_if_exists(driver)
        time.sleep(SCROLL_SLEEP)

    filtered = []
    for href, text in collected.items():
        filtered.append({"href": href, "card_text": text})

    return filtered


# =========================
# 메인 로직
# =========================

def crawl_gangnamunni_event_urls():
    df_cat = pd.read_excel(INPUT_EXCEL)
    print(f"📂 카테고리 로드: {df_cat.shape[0]} rows\n")

    has_big = COL_BIG in df_cat.columns
    has_mid = COL_MID in df_cat.columns

    if COL_SMALL not in df_cat.columns:
        raise ValueError(f"엑셀에 '{COL_SMALL}' 컬럼이 없습니다. 컬럼명을 확인해 주세요.")

    driver = setup_driver(headless=HEADLESS)
    all_rows = []

    try:
        for idx, row in df_cat.iterrows():
            small = str(row[COL_SMALL]).strip()
            if not small:
                continue

            big = str(row[COL_BIG]).strip() if has_big else None
            mid = str(row[COL_MID]).strip() if has_mid else None

            print(f"\n=== [{idx+1}/{len(df_cat)}] {big} / {mid} / {small} ===")

            for region_name, region_code in REGIONS.items():
                search_url = build_gangnamunni_search_url(small, region_code)
                print(f"🔍 [{region_name}] 검색 URL: {search_url}")

                # 1) 페이지 로드
                loaded = False
                for attempt in range(3):
                    try:
                        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
                        driver.get(search_url)
                        time.sleep(WAIT_AFTER_LOAD)
                        loaded = True
                        break
                    except TimeoutException:
                        print(f"⏰ 검색 페이지 로드 지연 — 재시도 {attempt+1}/3")
                        try:
                            driver.execute_script("window.stop();")
                        except Exception:
                            pass
                        time.sleep(2)

                if not loaded:
                    print(f"🚫 [{region_name}] 검색 페이지 로드 실패 → 스킵")
                    continue

                # 2) 카드 뜰 때까지 대기
                if not wait_for_any_card(driver, timeout=8):
                    print(f"   ❌ [{region_name}] 카드가 하나도 안 뜸 → 스킵")
                    continue

                # 3) 이벤트 총 건수 파싱
                total_events = get_event_count(driver)
                print(f"   ℹ [{region_name}] 검색 이벤트 수: {total_events}")

                # 4) '더보기' 버튼 1회 클릭 (있으면)
                click_load_more_once(driver)

                # 5) 카드 수집
                cards = collect_gangnamunni_event_cards(
                    driver, region_name, target_total=total_events
                )
                print(f"   🎯 '{small}' [{region_name}] 수집된 이벤트: {len(cards)}개")

                if not cards:
                    continue

                for c in cards:
                    href = c["href"]
                    if href.startswith("/"):
                        event_url = "https://www.gangnamunni.com" + href
                    else:
                        event_url = href

                    all_rows.append({
                        "대분류": big,
                        "중분류": mid,
                        "소분류": small,
                        "region": region_name,
                        "gangnamunni_search_url": search_url,
                        "search_event_count": total_events,
                        "event_url": event_url,
                        "card_text": c["card_text"],
                    })

            # 소분류 하나 끝날 때마다 중간 저장 (파일 열려 있으면 스킵)
            tmp = pd.DataFrame(all_rows)
            try:
                tmp.to_excel("gangnamunni_event_urls_tmp.xlsx",
                             index=False, engine="openpyxl")
                print(f"💾 중간 저장: gangnamunni_event_urls_tmp.xlsx "
                      f"(현재 {tmp.shape[0]}행)")
            except PermissionError:
                print("⚠ 중간 저장 실패: gangnamunni_event_urls_tmp.xlsx 가 열려 있습니다. "
                      "엑셀을 닫고 다시 실행하면 저장됩니다.")

    finally:
        driver.quit()

    df_out = pd.DataFrame(all_rows)
    df_out.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\n✅ 완료 → {OUTPUT_EXCEL} (총 {df_out.shape[0]}행)")

    return df_out


if __name__ == "__main__":
    crawl_gangnamunni_event_urls()


📂 카테고리 로드: 223 rows


=== [1/223] 코성형 / 코기능교정 / 비밸브협착증수술 ===
🔍 [서울] 검색 URL: https://www.gangnamunni.com/events?q=%EB%B9%84%EB%B0%B8%EB%B8%8C%ED%98%91%EC%B0%A9%EC%A6%9D%EC%88%98%EC%88%A0&district=N11&s=HIGHEST_RATING_COUNT
   ℹ [서울] 검색 이벤트 수: 51
   → '더보기' 버튼 클릭 완료
   ▸ Scroll 001 / cards: 40 (+40) [서울]
   ▸ Scroll 002 / cards: 40 (+0) [서울]
   ▸ Scroll 003 / cards: 40 (+0) [서울]
   ▸ Scroll 004 / cards: 40 (+0) [서울]
   ▸ Scroll 005 / cards: 40 (+0) [서울]
   ▸ Scroll 006 / cards: 51 (+11) [서울]
   🎯 필요 개수(51) 이상 수집 완료 → 스크롤 중단
   🎯 '비밸브협착증수술' [서울] 수집된 이벤트: 51개
🔍 [경기] 검색 URL: https://www.gangnamunni.com/events?q=%EB%B9%84%EB%B0%B8%EB%B8%8C%ED%98%91%EC%B0%A9%EC%A6%9D%EC%88%98%EC%88%A0&district=N12&s=HIGHEST_RATING_COUNT
   ℹ [경기] 검색 이벤트 수: 2
   (i) '더보기' 버튼을 찾지 못함 → 클릭 생략
   ▸ Scroll 001 / cards: 2 (+2) [경기]
   🎯 필요 개수(2) 이상 수집 완료 → 스크롤 중단
   🎯 '비밸브협착증수술' [경기] 수집된 이벤트: 2개
⚠ 중간 저장 실패: gangnamunni_event_urls_tmp.xlsx 가 열려 있습니다. 엑셀을 닫고 다시 실행하면 저장됩니다.

=== [2/223] 코성형 / 코기능교정 / 코골이치료 ===
🔍 [서울] 검